# GPT-2 (124M) Model from Scratch (with HF `accelerate` & Pre-trained Weight Loader)

This notebook contains the complete reproduction of Andrej Karpathy's GPT-2 (124M) architecture, including:
1. **`GPT2Config` Dataclass:** Standard GPT-2 124M parameters ($1024$ sequence length, $50,257$ vocab, $768$ embedding dim, $12$ heads, $12$ layers).
2. **FlashAttention `CausalSelfAttention`:** Fused CUDA kernel attention with 4D tensor reshaping.
3. **Pre-LN Architecture & GELU Activation:** Standard GPT-2 residual blocks.
4. **Weight Tying & Residual Scaling:** `lm_head.weight = wte.weight` and $\frac{1}{\sqrt{2N}}$ parameter initialization.
5. **`from_pretrained('gpt2')` Loader:** Seamlessly load official OpenAI pre-trained weights into our scratch PyTorch model.
6. **FineWeb Training & Evaluation:** FP16 training loop using Hugging Face **`accelerate`**.
7. **Top-K Autoregressive Text Generation:** Text sampling pipeline.

In [18]:
# ==========================================
# 1. IMPORTS & ACCELERATOR SETUP (REDUCED MICRO BATCH)
# ==========================================
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from transformers import AutoTokenizer, GPT2LMHeadModel
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader
from accelerate import Accelerator
from accelerate.utils import release_memory
from tqdm.notebook import tqdm

# Initialize Accelerator with FP16 mixed precision & Gradient Accumulation
GRAD_ACCUM_STEPS = 8
accelerator = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=GRAD_ACCUM_STEPS)
accelerator.print(f"Accelerator initialized on device: {accelerator.device} | Grad Accum Steps: {GRAD_ACCUM_STEPS}")

Accelerator initialized on device: cuda | Grad Accum Steps: 8


In [19]:
# ==========================================
# 2. GPT-2 CONFIGURATION DATACLASS (RAM & VRAM SAFE)
# ==========================================
@dataclass
class GPT2Config:
    num_documents: int = 5000   # Fast, RAM-safe FineWeb slice (~5 Million tokens)
    B: int = 16                 # Micro-batch size per pass
    T: int = 1024               # Sequence length (T)
    C: int = 768                # Embedding dimension (C)
    n_head: int = 12            # Number of self-attention heads
    n_layer: int = 12           # Number of stacked Transformer blocks
    dropout: float = 0.1        # 10% Dropout regularization
    learning_rate: float = 6e-4 # Max learning rate
    min_lr: float = 6e-5        # Min learning rate for cosine schedule
    warmup_steps: int = 50      # Warmup steps
    max_steps: int = 1000       # Total training steps
    eval_interval: int = 100    # Evaluate dev loss every 100 steps
    vocab_size: int = 49152     # Automatically populated by SmolLM Tokenizer
    head_dim: int = field(init=False)

    def __post_init__(self):
        assert self.C % self.n_head == 0, f"Embedding dim C ({self.C}) must be divisible by n_head ({self.n_head})"
        self.head_dim = self.C // self.n_head

config = GPT2Config()
effective_batch = config.B * GRAD_ACCUM_STEPS
tokens_per_step = effective_batch * config.T
print(f"GPT-2 RAM-Safe Configuration:\n  Documents          : {config.num_documents:,}\n  Micro Batch (B)    : {config.B}\n  Grad Accum Steps   : {GRAD_ACCUM_STEPS}\n  Effective Batch    : {effective_batch} sequences\n  Tokens / Step      : {tokens_per_step:,} tokens\n  Max Steps          : {config.max_steps:,}")

GPT-2 RAM-Safe Configuration:
  Documents          : 5,000
  Micro Batch (B)    : 16
  Grad Accum Steps   : 8
  Effective Batch    : 128 sequences
  Tokens / Step      : 131,072 tokens
  Max Steps          : 1,000


In [20]:
# ==========================================
# 3. GPT-2 ARCHITECTURE COMPONENTS (FLASHATTENTION + MODERN SWIGLU MLP)
# ==========================================
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.C % config.n_head == 0
        # Key, Query, Value projections in 1 batched linear layer
        self.c_attn = nn.Linear(config.C, 3 * config.C)
        # Output projection
        self.c_proj = nn.Linear(config.C, config.C)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        
        self.n_head = config.n_head
        self.C = config.C
        self.head_dim = config.head_dim
        self.dropout_p = config.dropout

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.C, dim=2)
        
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        dropout_p = self.dropout_p if self.training else 0.0
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout_p)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLUMLP(nn.Module):
    """ Modern SwiGLU FeedForward Network (used in LLaMA 3, Qwen 2.5, & Mistral) """
    def __init__(self, config):
        super().__init__()
        # Standard SwiGLU hidden dim scaling: (8/3) * C rounded to multiple of 64
        hidden_dim = int(2 * (4 * config.C) / 3)
        hidden_dim = 64 * ((hidden_dim + 63) // 64)
        
        self.w1 = nn.Linear(config.C, hidden_dim, bias=False) # Content projection
        self.w2 = nn.Linear(hidden_dim, config.C, bias=False) # Output projection
        self.w3 = nn.Linear(config.C, hidden_dim, bias=False) # Gate projection
        self.w2.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        # Gated Swish Activation: F.silu(w1(x)) * w3(x)
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.C)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.C)
        self.mlp  = SwiGLUMLP(config) # Upgraded to SwiGLU MLP

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.C),
            wpe = nn.Embedding(config.T, config.C),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.C),
        ))
        self.lm_head = nn.Linear(config.C, config.vocab_size, bias=False)
        
        # Weight Tying
        self.transformer.wte.weight = self.lm_head.weight
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.T, f"Cannot forward sequence length {T}, model block size is {self.config.T}"
        
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
            
        return logits, loss

In [21]:
# ==========================================
# 4. MULTI-SHARD BINARY DATASET LOADER (ZERO-RAM NP.MEMMAP)
# ==========================================
import glob
import numpy as np

shards_dir = "shards"
bin_shards = sorted(glob.glob(os.path.join(shards_dir, "fineweb_shard_*.bin")))

if not bin_shards:
    # Fallback to single shard if multi-shard directory is empty
    if os.path.exists("fineweb_tokens.bin"):
        bin_shards = ["fineweb_tokens.bin"]
    else:
        accelerator.print(f"No binary shards found in '{shards_dir}/'!")
        accelerator.print("Run 'python tokenize_dataset.py' to generate FineWeb binary dataset shards.")

if bin_shards:
    accelerator.print(f"Found {len(bin_shards)} binary shard(s): {bin_shards[:3]}...")
    
    # Memory-map all binary shards into a single contiguous array using Zero RAM!
    shard_tensors = []
    total_tokens_loaded = 0
    for shard in bin_shards:
        tokens_np = np.memmap(shard, dtype=np.uint16, mode='r')
        total_tokens_loaded += len(tokens_np)
        shard_tensors.append(torch.from_numpy(tokens_np.astype(np.int64)))
        
    data_tensor = torch.cat(shard_tensors, dim=0)
    accelerator.print(f"Loaded {total_tokens_loaded:,} tokens via Zero-RAM np.memmap!")
    
    unfolded = data_tensor.unfold(dimension=0, size=config.T + 1, step=256)
    
    n1 = int(0.8 * unfolded.shape[0])
    n2 = int(0.9 * unfolded.shape[0])
    
    Xtr, Ytr = unfolded[:n1, :config.T], unfolded[:n1, 1:config.T + 1]
    Xdev, Ydev = unfolded[n1:n2, :config.T], unfolded[n1:n2, 1:config.T + 1]
    Xte, Yte = unfolded[n2:, :config.T], unfolded[n2:, 1:config.T + 1]
    
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=config.B, shuffle=True)
    dev_loader   = DataLoader(TensorDataset(Xdev, Ydev), batch_size=config.B)
    test_loader  = DataLoader(TensorDataset(Xte, Yte), batch_size=config.B)
    
    train_loader, dev_loader, test_loader = accelerator.prepare(
        train_loader, dev_loader, test_loader
    )
    
    accelerator.print(f"\nTrain dataset: {len(Xtr):,} sequences | Dev dataset: {len(Xdev):,} sequences | Test dataset: {len(Xte):,} sequences")

Found 1 binary shard(s): ['fineweb_tokens.bin']...
Loaded 14,500,102 tokens via Zero-RAM np.memmap!

Train dataset: 45,310 sequences | Dev dataset: 5,664 sequences | Test dataset: 5,664 sequences


In [22]:
# ==========================================
# 5. MODEL INSTANTIATION, DEV LOSS MAPPING & EXTENDED TRAINING LOOP
# ==========================================
import matplotlib.pyplot as plt

model = GPT2(config)

# Print detailed parameter breakdown
wte_params = sum(p.numel() for p in model.transformer.wte.parameters())
wpe_params = sum(p.numel() for p in model.transformer.wpe.parameters())
blocks_params = sum(p.numel() for p in model.transformer.h.parameters())
ln_f_params = sum(p.numel() for p in model.transformer.ln_f.parameters())
total_params = sum(p.numel() for p in model.parameters())

accelerator.print("\n===========================================")
accelerator.print("  GPT-2 (124M) DETAILED PARAMETER BREAKDOWN")
accelerator.print("===========================================")
accelerator.print(f"  Token Embeddings (wte)    : {wte_params:,} ({wte_params/total_params*100:.1f}%)")
accelerator.print(f"  Position Embeddings (wpe) : {wpe_params:,} ({wpe_params/total_params*100:.1f}%)")
accelerator.print(f"  12 Transformer Blocks (h) : {blocks_params:,} ({blocks_params/total_params*100:.1f}%)")
accelerator.print(f"  Final LayerNorm (ln_f)    : {ln_f_params:,} (<0.1%)")
accelerator.print(f"  Output LM Head (lm_head)  : 0 (Weight Tied with wte)")
accelerator.print("-------------------------------------------")
accelerator.print(f"  TOTAL TRAINABLE PARAMETERS: {total_params:,}")
accelerator.print("===========================================\n")

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.1, betas=(0.9, 0.95))

# Prepare model and optimizer with Accelerate!
model, optimizer = accelerator.prepare(model, optimizer)

accelerator.print(f"Starting training for {config.max_steps} steps ({tokens_per_step * config.max_steps:,} total tokens)...")

model.train()
step = 0
train_history = []
dev_history = []
step_history = []

pbar = tqdm(total=config.max_steps, desc="Training GPT-2 (124M)")

while step < config.max_steps:
    for xb, yb in train_loader:
        if step >= config.max_steps:
            break
            
        # Cosine Learning Rate Schedule with Warmup
        if step < config.warmup_steps:
            lr = config.learning_rate * (step + 1) / config.warmup_steps
        else:
            decay_ratio = (step - config.warmup_steps) / (config.max_steps - config.warmup_steps)
            coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
            lr = config.min_lr + coeff * (config.learning_rate - config.min_lr)
            
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
            
        # Use accelerator.accumulate for automated gradient accumulation
        with accelerator.accumulate(model):
            logits, loss = model(xb, yb)
            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            
        if accelerator.sync_gradients:
            step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{lr:.2e}"})
            
            # Evaluate Dev Loss periodically
            if step % config.eval_interval == 0 or step == config.max_steps:
                model.eval()
                total_dev_loss = 0.0
                dev_batches = 0
                with torch.no_grad():
                    for xb_dev, yb_dev in dev_loader:
                        _, dev_loss = model(xb_dev, yb_dev)
                        total_dev_loss += dev_loss.item()
                        dev_batches += 1
                        if dev_batches >= 20: # Fast evaluation sample
                            break
                            
                avg_dev_loss = total_dev_loss / dev_batches
                train_history.append(loss.item())
                dev_history.append(avg_dev_loss)
                step_history.append(step)
                
                accelerator.print(f"Step {step:4d}/{config.max_steps} | Train Loss: {loss.item():.4f} | Dev Loss: {avg_dev_loss:.4f} | LR: {lr:.2e}")
                model.train()

pbar.close()
accelerator.print("Training Complete!")

# Plot Train Loss vs Dev Loss Curve
if accelerator.is_main_process:
    plt.figure(figsize=(9, 5))
    plt.plot(step_history, train_history, label="Train Loss", marker="o")
    plt.plot(step_history, dev_history, label="Dev Loss", marker="s")
    plt.xlabel("Step")
    plt.ylabel("Cross Entropy Loss")
    plt.title("GPT-2 (124M) Training & Dev Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.show()


  GPT-2 (124M) DETAILED PARAMETER BREAKDOWN
  Token Embeddings (wte)    : 37,748,736 (30.6%)
  Position Embeddings (wpe) : 786,432 (0.6%)
  12 Transformer Blocks (h) : 85,008,384 (68.8%)
  Final LayerNorm (ln_f)    : 1,536 (<0.1%)
  Output LM Head (lm_head)  : 0 (Weight Tied with wte)
-------------------------------------------
  TOTAL TRAINABLE PARAMETERS: 123,545,088

Starting training for 1000 steps (131,072,000 total tokens)...


Training GPT-2 (124M):   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ==========================================
# 6. AUTOREGRESSIVE TEXT GENERATION WITH TOP-K SAMPLING
# ==========================================
def generate_text(start_str="Hello, I am a language model,", max_new_tokens=50, temperature=0.7, top_k=50):
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.eval()
    device = accelerator.device
    input_indices = tokenizer.encode(start_str)
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context_indices = input_indices[-config.T:]
            x = torch.tensor([context_indices], dtype=torch.long, device=device)
            
            logits, _ = unwrapped_model(x)
            next_token_logits = logits[:, -1, :] / temperature
            
            top_k_logits, top_k_indices = torch.topk(next_token_logits, top_k, dim=-1)
            probs = F.softmax(top_k_logits, dim=-1)
            
            idx = torch.multinomial(probs, num_samples=1)
            next_token = torch.gather(top_k_indices, -1, idx).item()
            input_indices.append(next_token)
            
    return tokenizer.decode(input_indices)

accelerator.print("--- GENERATED SAMPLE TEXT ---")
accelerator.print(generate_text(start_str="Hello, I am a language model,", max_new_tokens=50, temperature=0.7, top_k=50))

--- GENERATED SAMPLE TEXT ---
Hello, I am a language model, and you can see my own is my own. I know most of you in the world. I am looking for your children by now. I am looking for the kids to come to this.

You have a 3D and 4


In [ ]:
# ==========================================
# 7. VRAM CLEANUP
# ==========================================
model, optimizer = release_memory(model, optimizer)
accelerator.free_memory()
if torch.cuda.is_available():
    print(f"VRAM Released! Current Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

VRAM Released! Current Allocated: 7955.66 MB


What Makes SwiGLU Superior:
Multiplicative Gating Mechanism: Instead of a simple $x \rightarrow \text{GELU}(W_1 x) \rightarrow W_2 x$ pipeline, SwiGLU splits projection into two parallel branches:

Content Branch: $W_1 x$
Gate Branch: $W_3 x$ $$\text{SwiGLU}(x) = W_2 \left( \text{SiLU}(W_1 x) \times W_3 x \right)$$ This allows the network to dynamically gate and filter which features pass through!
Parameter Scaling: To keep total parameters identical to standard MLPs, the hidden dimension is scaled to $\frac{8}{3} C$ (rounded to a Tensor Core multiple of 64): $$\text{hidden_dim} = \frac{8}{3} \times 768 = \mathbf{2,048}$$